PHASE4: DEFINE FX STRESS

Step7: Define a Target

In [2]:
import pandas as pd

# ============================================================
# STEP 7.1 - Volatility thresholds
# ============================================================

df_volatility = pd.read_csv("../data/processed/fx_volatility_30d.csv")
df_returns = pd.read_csv("../data/processed/fx_returns.csv")

volatility_cols = [
    "CNY_vol30",
    "EUR_vol30",
    "INR_vol30",
    "ZAR_vol30",
    "TRY_vol30",
    "JPY_vol30",
    "GBP_vol30",
    "BRL_vol30",
    "AUD_vol30"
]

# Calculate percentile thresholds
vol_percentiles = (
    df_volatility[volatility_cols]
    .quantile([0.75, 0.90, 0.95, 0.99])
    .T
)

# Display as percentages
print(vol_percentiles.map(lambda x: f"{x:.2%}"))

            0.75   0.90   0.95   0.99
CNY_vol30  0.26%  0.36%  0.40%  0.62%
EUR_vol30  0.55%  0.69%  0.77%  0.92%
INR_vol30  0.36%  0.43%  0.46%  0.64%
ZAR_vol30  1.05%  1.26%  1.43%  1.63%
TRY_vol30  0.84%  1.22%  1.75%  4.79%
JPY_vol30  0.67%  0.84%  0.94%  1.23%
GBP_vol30  0.60%  0.73%  0.86%  1.52%
BRL_vol30  1.11%  1.35%  1.49%  2.01%
AUD_vol30  0.71%  0.88%  0.99%  1.25%


### Step 7.2 — Standardizing Volatility Across Currencies

**Objective:** Make volatility levels comparable across the nine currencies by measuring how unusual each currency's volatility is relative to its own historical distribution.

In Step 7.1, we found substantial differences in historical volatility thresholds. For example, the 95th-percentile volatility is **0.40% for CNY** but **1.75% for TRY**. Therefore, applying the same absolute volatility threshold to every currency could misrepresent the level of risk.

To address this, we convert each currency's 30-day rolling volatility into a **historical percentile rank** between 0 and 1.

- A percentile rank near **0.50** indicates volatility around the middle of its historical distribution.
- A rank near **0.90** indicates unusually high volatility.
- A rank near **0.95 or above** indicates volatility near the upper tail of its historical distribution.

This transformation allows us to compare currencies on a common relative scale, regardless of their typical volatility levels.

**Expected outcome:** A new dataset containing standardized volatility percentile ranks for all nine currencies. These comparable measures will provide the foundation for identifying periods when elevated volatility occurs across multiple currencies simultaneously.

**Methodological limitation:** Using the full historical sample to calculate percentile ranks incorporates information from future observations when evaluating past dates. This is acceptable for exploratory historical analysis, but a past-only approach will be required when developing a real-time early-warning system.

In [8]:

# ============================================================
# STEP 7.2 - Standardize volatality across currencies
# ============================================================

# Create a new dataframe without modifying original volatility data
df_vol_percentile = df_volatility[["Day"]].copy()

# Calculate historical percentile ranks for each currency
for col in volatility_cols:
    new_col = col.replace("_vol30", "_vol_percentile")
    df_vol_percentile[new_col] = df_volatility[col].rank(pct=True)

# Identify percentile columns
percentile_cols = df_vol_percentile.columns.drop("Day")

# Display summary statistics
summary = df_vol_percentile[percentile_cols].describe()
summary_display = summary.map(lambda x: f"{x:.2%}")

# Display observation counts as integers, not percentages
summary_display.loc["count"] = summary.loc["count"].map(
    lambda x: f"{x:,.0f}"
)

print("Summary statistics:")
print(summary_display)

# Save processed dataset
import os

os.makedirs("../data/processed", exist_ok=True)

df_vol_percentile.to_csv(
    "../data/processed/fx_volatility_percentiles.csv",
    index=False
)

print("\nDataset saved successfully:")
print("../data/processed/fx_volatility_percentiles.csv")

Summary statistics:
      CNY_vol_percentile EUR_vol_percentile INR_vol_percentile  \
count              2,966              2,966              2,966   
mean              50.02%             50.02%             50.02%   
std               28.87%             28.87%             28.87%   
min                0.03%              0.03%              0.03%   
25%               25.03%             25.03%             25.03%   
50%               50.02%             50.02%             50.02%   
75%               75.01%             75.01%             75.01%   
max              100.00%            100.00%            100.00%   

      ZAR_vol_percentile TRY_vol_percentile JPY_vol_percentile  \
count              2,966              2,966              2,966   
mean              50.02%             50.02%             50.02%   
std               28.87%             28.87%             28.87%   
min                0.03%              0.03%              0.03%   
25%               25.03%             25.03%            